# Journal Extension J5 — Representation & Prediction Complementarity
## Original 83.95% Thesis Protocol

This notebook explains **why heterogeneous SSL helps**. This FINAL FIXED version also correctly loads J2 object-dtype image IDs.

It uses the exact original thesis validation/test protocol and compares:

- **SimCLR**
- **Rotation SSL**
- **DINOv2 ViT-S/14**
- **Final proposed ensemble**

No training is performed.

## Analyses

### Representation-level
1. Pairwise **linear CKA**:
   - SimCLR ↔ Rotation
   - SimCLR ↔ DINOv2
   - Rotation ↔ DINOv2
2. CKA permutation significance test.
3. UMAP visualization for each representation.
4. Representation quality:
   - silhouette score
   - 10-NN label purity
   - Fisher discriminant ratio

### Prediction-level
1. Pairwise disagreement rate.
2. Q-statistic.
3. Double-fault rate.
4. Both-correct / only-A-correct / only-B-correct / both-wrong.
5. Class-wise unique-correct cases.
6. Errors rescued by the final proposed model.
7. Final vs SimCLR / Rotation / DINO per-class rescue counts.

## Required Kaggle inputs

Attach:

### A. Original full thesis pipeline output — REQUIRED
Saved output of `last-part-of-the-previous-one`.

It must contain:
- `rotation_ssl_paperlike/fusion_features_multiseed_rep_v2/val_features.npz`
- `rotation_ssl_paperlike/fusion_features_multiseed_rep_v2/test_features.npz`
- original SimCLR probabilities
- original Rotation prediction CSV

### B. Confidence-gated ensemble output — REQUIRED
Needed for the frozen final probabilities.

### C. Executed J2 DINOv2 output — REQUIRED
Must contain:
- `dinov2_representative_val_features.npz`
- `dinov2_representative_test_features.npz`
- `dinov2_original_split_multiseed_probabilities.npz`

### D. J1 protocol-lock output — RECOMMENDED
Used for image-ID/order verification when available.

## Compute
CPU is sufficient. Internet is recommended only if `umap-learn` is not already installed.

In [1]:
# ============================================================
# CELL 0 — OPTIONAL UMAP DEPENDENCY
# ============================================================
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("umap") is None:
    print("umap-learn is not installed. Installing it...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "umap-learn",
        ],
        check=True,
    )
else:
    print("umap-learn already available.")

umap-learn already available.


In [2]:
# ============================================================
# CELL 1 — IMPORTS / CONFIG
# ============================================================
import gc
import json
import math
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

import umap

KAGGLE_INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

OUTPUT_DIR = WORKING / "j5_original_split_complementarity"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
AUDIT_DIR = OUTPUT_DIR / "audit"
EMBED_DIR = OUTPUT_DIR / "embeddings"

for d in [OUTPUT_DIR, TABLE_DIR, FIG_DIR, AUDIT_DIR, EMBED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["MEL","NV","BCC","AK","BKL","DF","VASC","SCC"]
NUM_CLASSES = 8

EXPECTED_VAL_N = 2533
EXPECTED_TEST_N = 2534

RANDOM_STATE = 42

CKA_PERMUTATIONS = 200
CKA_PERMUTATION_SUBSET = 800

UMAP_MAX_SAMPLES = 2533
UMAP_PCA_DIM = 50

QUALITY_PCA_DIM = 64
KNN_K = 10

print("=" * 96)
print("J5 — ORIGINAL-SPLIT REPRESENTATION & PREDICTION COMPLEMENTARITY")
print("=" * 96)
print("CKA permutations:", CKA_PERMUTATIONS)
print("Output:", OUTPUT_DIR)

J5 — ORIGINAL-SPLIT REPRESENTATION & PREDICTION COMPLEMENTARITY
CKA permutations: 200
Output: /kaggle/working/j5_original_split_complementarity


In [3]:
# ============================================================
# CELL 2 — GENERAL HELPERS
# ============================================================

def find_all(filename):
    return sorted(
        KAGGLE_INPUT.rglob(filename),
        key=lambda p: (len(p.parts), str(p))
    )

def normalize_probabilities(probabilities):
    p = np.asarray(probabilities, dtype=np.float64)

    if p.ndim != 2 or p.shape[1] != NUM_CLASSES:
        raise ValueError(
            f"Expected probability shape (N,{NUM_CLASSES}), got {p.shape}"
        )

    p = np.nan_to_num(
        p,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    p = np.clip(p, 1e-12, None)
    p /= p.sum(axis=1, keepdims=True)

    return p.astype(np.float32)

def probability_columns(df):
    cols = [
        c for c in df.columns
        if c.startswith("prob_class_")
        and c.rsplit("_", 1)[-1].isdigit()
    ]
    if len(cols) == NUM_CLASSES:
        return sorted(
            cols,
            key=lambda c: int(c.rsplit("_", 1)[-1])
        )

    cols = [
        c for c in df.columns
        if c.startswith("prob_")
        and len(c.split("_")) >= 2
        and c.split("_")[1].isdigit()
    ]
    if len(cols) == NUM_CLASSES:
        return sorted(
            cols,
            key=lambda c: int(c.split("_")[1])
        )

    exact = [
        f"prob_{i}"
        for i in range(NUM_CLASSES)
        if f"prob_{i}" in df.columns
    ]
    if len(exact) == NUM_CLASSES:
        return exact

    raise ValueError(
        "Could not detect exactly 8 probability columns."
    )

def label_column(df):
    for c in ["y_true", "true_label", "label"]:
        if c in df.columns:
            return c

    raise ValueError(
        "True-label column not found."
    )

def load_prediction_csv(path):
    df = pd.read_csv(path)

    y_col = label_column(df)
    p_cols = probability_columns(df)

    y = df[y_col].to_numpy(np.int32)

    p = normalize_probabilities(
        df[p_cols].to_numpy(np.float32)
    )

    return y, p

def l2_normalize_rows(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float64)

    norms = np.linalg.norm(
        x,
        axis=1,
        keepdims=True,
    )

    return (
        x
        / np.clip(norms, eps, None)
    ).astype(np.float32)

print("Helpers ready.")

Helpers ready.


In [4]:
# ============================================================
# CELL 3 — FIND ORIGINAL FEATURE FILES
# ============================================================

val_feature_hits = find_all("val_features.npz")
test_feature_hits = find_all("test_features.npz")

ORIGINAL_VAL_FEATURES = None
ORIGINAL_TEST_FEATURES = None

for p in val_feature_hits:
    if "fusion_features_multiseed_rep_v2" not in str(p):
        continue

    try:
        with np.load(p) as z:
            if not {
                "simclr_features",
                "rotation_features",
                "labels",
            }.issubset(z.files):
                continue

            if len(z["labels"]) != EXPECTED_VAL_N:
                continue

        ORIGINAL_VAL_FEATURES = p
        break

    except Exception:
        continue

for p in test_feature_hits:
    if "fusion_features_multiseed_rep_v2" not in str(p):
        continue

    try:
        with np.load(p) as z:
            if not {
                "simclr_features",
                "rotation_features",
                "labels",
            }.issubset(z.files):
                continue

            if len(z["labels"]) != EXPECTED_TEST_N:
                continue

        ORIGINAL_TEST_FEATURES = p
        break

    except Exception:
        continue

if ORIGINAL_VAL_FEATURES is None or ORIGINAL_TEST_FEATURES is None:
    raise FileNotFoundError(
        "Original SimCLR/Rotation fusion feature files were not found. "
        "Attach the saved output of `last-part-of-the-previous-one`."
    )

with np.load(ORIGINAL_VAL_FEATURES) as z:
    Y_VAL = np.asarray(z["labels"], dtype=np.int32)
    SIM_VAL_FEATURES = np.asarray(
        z["simclr_features"],
        dtype=np.float32,
    )
    ROT_VAL_FEATURES = np.asarray(
        z["rotation_features"],
        dtype=np.float32,
    )

with np.load(ORIGINAL_TEST_FEATURES) as z:
    Y_TEST = np.asarray(z["labels"], dtype=np.int32)
    SIM_TEST_FEATURES = np.asarray(
        z["simclr_features"],
        dtype=np.float32,
    )
    ROT_TEST_FEATURES = np.asarray(
        z["rotation_features"],
        dtype=np.float32,
    )

print("Original VAL features:", ORIGINAL_VAL_FEATURES)
print("Original TEST features:", ORIGINAL_TEST_FEATURES)

print("\nShapes")
print("SimCLR VAL  :", SIM_VAL_FEATURES.shape)
print("Rotation VAL:", ROT_VAL_FEATURES.shape)
print("SimCLR TEST :", SIM_TEST_FEATURES.shape)
print("Rotation TEST:", ROT_TEST_FEATURES.shape)

if len(Y_VAL) != EXPECTED_VAL_N:
    raise RuntimeError("Wrong original validation size.")

if len(Y_TEST) != EXPECTED_TEST_N:
    raise RuntimeError("Wrong original test size.")

if not np.array_equal(
    np.unique(Y_VAL),
    np.arange(NUM_CLASSES),
):
    raise RuntimeError("Validation labels are not 0..7.")

if not np.array_equal(
    np.unique(Y_TEST),
    np.arange(NUM_CLASSES),
):
    raise RuntimeError("Test labels are not 0..7.")

Original VAL features: /kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/fusion_features_multiseed_rep_v2/val_features.npz
Original TEST features: /kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/fusion_features_multiseed_rep_v2/test_features.npz

Shapes
SimCLR VAL  : (2533, 2048)
Rotation VAL: (2533, 2048)
SimCLR TEST : (2534, 2048)
Rotation TEST: (2534, 2048)


In [5]:
# ============================================================
# CELL 4 — LOAD DINO REPRESENTATIVE FEATURES
# FIX: J2 saved image_id as an object-dtype NumPy array.
# The J2 files are our own trusted experiment artifacts, so this
# cell enables allow_pickle=True only for these two feature NPZs.
# Numeric features/probabilities/labels are still explicitly cast.
# ============================================================

dino_val_hits = find_all(
    "dinov2_representative_val_features.npz"
)

dino_test_hits = find_all(
    "dinov2_representative_test_features.npz"
)

DINO_VAL_NPZ = None
DINO_TEST_NPZ = None

# First identify the correctly aligned J2 validation feature file.
for p in dino_val_hits:
    try:
        with np.load(p, allow_pickle=True) as z:
            if not {
                "labels",
                "features",
                "probabilities",
            }.issubset(z.files):
                continue

            labels = np.asarray(
                z["labels"],
                dtype=np.int32,
            )

            if not np.array_equal(
                labels,
                Y_VAL,
            ):
                continue

        DINO_VAL_NPZ = p
        break

    except Exception:
        continue

# Then identify the correctly aligned J2 test feature file.
for p in dino_test_hits:
    try:
        with np.load(p, allow_pickle=True) as z:
            if not {
                "labels",
                "features",
                "probabilities",
            }.issubset(z.files):
                continue

            labels = np.asarray(
                z["labels"],
                dtype=np.int32,
            )

            if not np.array_equal(
                labels,
                Y_TEST,
            ):
                continue

        DINO_TEST_NPZ = p
        break

    except Exception:
        continue

if DINO_VAL_NPZ is None or DINO_TEST_NPZ is None:
    raise FileNotFoundError(
        "J2 representative DINO feature files were not found/aligned. "
        "Attach the executed J2 output containing both representative "
        "validation and test feature NPZ files."
    )

# Load the trusted J2 artifact.
with np.load(
    DINO_VAL_NPZ,
    allow_pickle=True,
) as z:
    DINO_VAL_FEATURES = np.asarray(
        z["features"],
        dtype=np.float32,
    )

    DINO_VAL_REP_PROB = normalize_probabilities(
        np.asarray(
            z["probabilities"],
            dtype=np.float32,
        )
    )

    DINO_VAL_IDS = None
    if "image_id" in z.files:
        DINO_VAL_IDS = np.asarray(
            z["image_id"],
            dtype=object,
        ).astype(str)

    DINO_REP_SEED = None
    if "seed" in z.files:
        DINO_REP_SEED = int(
            np.asarray(
                z["seed"]
            ).ravel()[0]
        )

# Load the trusted J2 test artifact.
with np.load(
    DINO_TEST_NPZ,
    allow_pickle=True,
) as z:
    DINO_TEST_FEATURES = np.asarray(
        z["features"],
        dtype=np.float32,
    )

    DINO_TEST_REP_PROB = normalize_probabilities(
        np.asarray(
            z["probabilities"],
            dtype=np.float32,
        )
    )

    DINO_TEST_IDS = None
    if "image_id" in z.files:
        DINO_TEST_IDS = np.asarray(
            z["image_id"],
            dtype=object,
        ).astype(str)

# Hard numerical-shape checks.
if DINO_VAL_FEATURES.shape[0] != EXPECTED_VAL_N:
    raise RuntimeError(
        f"DINO validation feature count mismatch: "
        f"{DINO_VAL_FEATURES.shape[0]} != {EXPECTED_VAL_N}"
    )

if DINO_TEST_FEATURES.shape[0] != EXPECTED_TEST_N:
    raise RuntimeError(
        f"DINO test feature count mismatch: "
        f"{DINO_TEST_FEATURES.shape[0]} != {EXPECTED_TEST_N}"
    )

if DINO_VAL_REP_PROB.shape != (EXPECTED_VAL_N, NUM_CLASSES):
    raise RuntimeError(
        f"Unexpected DINO validation probability shape: "
        f"{DINO_VAL_REP_PROB.shape}"
    )

if DINO_TEST_REP_PROB.shape != (EXPECTED_TEST_N, NUM_CLASSES):
    raise RuntimeError(
        f"Unexpected DINO test probability shape: "
        f"{DINO_TEST_REP_PROB.shape}"
    )

if not np.isfinite(DINO_VAL_FEATURES).all():
    raise RuntimeError("DINO validation features contain NaN/Inf.")

if not np.isfinite(DINO_TEST_FEATURES).all():
    raise RuntimeError("DINO test features contain NaN/Inf.")

print("DINO representative VAL file:")
print(" ", DINO_VAL_NPZ)

print("DINO representative TEST file:")
print(" ", DINO_TEST_NPZ)

print("DINO representative seed:", DINO_REP_SEED)
print("DINO VAL :", DINO_VAL_FEATURES.shape)
print("DINO TEST:", DINO_TEST_FEATURES.shape)

if DINO_VAL_IDS is not None:
    print("DINO VAL IDs loaded:", DINO_VAL_IDS.shape)

if DINO_TEST_IDS is not None:
    print("DINO TEST IDs loaded:", DINO_TEST_IDS.shape)

print("DINO representative feature loading: PASSED")

DINO representative VAL file:
  /kaggle/input/notebooks/anirbanroysudipto/journal-extension-j2/j2_dinov2_original_split/features/dinov2_representative_val_features.npz
DINO representative TEST file:
  /kaggle/input/notebooks/anirbanroysudipto/journal-extension-j2/j2_dinov2_original_split/features/dinov2_representative_test_features.npz
DINO representative seed: 42
DINO VAL : (2533, 384)
DINO TEST: (2534, 384)
DINO VAL IDs loaded: (2533,)
DINO TEST IDs loaded: (2534,)
DINO representative feature loading: PASSED


In [6]:
# ============================================================
# CELL 5 — OPTIONAL J1 IMAGE-ID ORDER VERIFICATION
# ============================================================

locked_hits = find_all(
    "original_80_10_10_locked_ids.csv"
)

ID_AUDIT = {
    "j1_locked_ids_found": False,
    "dino_val_id_order_matches_j1": None,
    "dino_test_id_order_matches_j1": None,
}

if locked_hits:
    locked_path = locked_hits[0]
    locked_df = pd.read_csv(locked_path)

    if {
        "split",
        "image_id",
        "label",
    }.issubset(locked_df.columns):

        locked_val = locked_df.loc[
            locked_df["split"] == "VAL"
        ].reset_index(drop=True)

        locked_test = locked_df.loc[
            locked_df["split"] == "TEST"
        ].reset_index(drop=True)

        ID_AUDIT["j1_locked_ids_found"] = True
        ID_AUDIT["locked_ids_source"] = str(locked_path)

        if DINO_VAL_IDS is not None:
            ID_AUDIT[
                "dino_val_id_order_matches_j1"
            ] = bool(
                np.array_equal(
                    DINO_VAL_IDS.astype(str),
                    locked_val["image_id"].astype(str).to_numpy(),
                )
            )

        if DINO_TEST_IDS is not None:
            ID_AUDIT[
                "dino_test_id_order_matches_j1"
            ] = bool(
                np.array_equal(
                    DINO_TEST_IDS.astype(str),
                    locked_test["image_id"].astype(str).to_numpy(),
                )
            )

        if not np.array_equal(
            locked_val["label"].to_numpy(np.int32),
            Y_VAL,
        ):
            raise RuntimeError(
                "Original feature validation labels do not align with J1."
            )

        if not np.array_equal(
            locked_test["label"].to_numpy(np.int32),
            Y_TEST,
        ):
            raise RuntimeError(
                "Original feature test labels do not align with J1."
            )

print(json.dumps(ID_AUDIT, indent=2))

with open(
    AUDIT_DIR / "feature_order_audit.json",
    "w",
) as f:
    json.dump(
        ID_AUDIT,
        f,
        indent=2,
    )

{
  "j1_locked_ids_found": true,
  "dino_val_id_order_matches_j1": true,
  "dino_test_id_order_matches_j1": true,
  "locked_ids_source": "/kaggle/input/notebooks/anirbanroysudipto/journal-extension-j1/j1_original_protocol_lock/tables/original_80_10_10_locked_ids.csv"
}


In [7]:
# ============================================================
# CELL 6 — LINEAR CKA
# ============================================================

def center_gram(k):
    k = np.asarray(k, dtype=np.float64)

    row_mean = k.mean(
        axis=1,
        keepdims=True,
    )

    col_mean = k.mean(
        axis=0,
        keepdims=True,
    )

    grand_mean = k.mean()

    return (
        k
        - row_mean
        - col_mean
        + grand_mean
    )

def linear_gram(features):
    x = np.asarray(
        features,
        dtype=np.float64,
    )

    x = x - x.mean(
        axis=0,
        keepdims=True,
    )

    return x @ x.T

def cka_from_centered_grams(kc, lc):
    numerator = np.sum(kc * lc)

    denominator = np.sqrt(
        np.sum(kc * kc)
        * np.sum(lc * lc)
    )

    if denominator <= 0:
        return np.nan

    return float(
        numerator / denominator
    )

def linear_cka(x, y):
    if len(x) != len(y):
        raise ValueError("CKA requires aligned samples.")

    k = center_gram(
        linear_gram(x)
    )

    l = center_gram(
        linear_gram(y)
    )

    return cka_from_centered_grams(
        k,
        l,
    )

REPRESENTATIONS = {
    "SimCLR": SIM_VAL_FEATURES,
    "Rotation": ROT_VAL_FEATURES,
    "DINOv2": DINO_VAL_FEATURES,
}

pair_names = [
    ("SimCLR", "Rotation"),
    ("SimCLR", "DINOv2"),
    ("Rotation", "DINOv2"),
]

cka_rows = []

for a, b in pair_names:
    score = linear_cka(
        REPRESENTATIONS[a],
        REPRESENTATIONS[b],
    )

    cka_rows.append({
        "representation_a": a,
        "representation_b": b,
        "linear_cka": score,
        "n_samples": len(Y_VAL),
        "split": "Validation",
    })

    print(
        f"{a} ↔ {b}: CKA = {score:.6f}"
    )

CKA_DF = pd.DataFrame(
    cka_rows
)

display(CKA_DF)

CKA_DF.to_csv(
    TABLE_DIR / "pairwise_linear_cka.csv",
    index=False,
)

SimCLR ↔ Rotation: CKA = 0.248448
SimCLR ↔ DINOv2: CKA = 0.606945
Rotation ↔ DINOv2: CKA = 0.145899


  representation_a representation_b  linear_cka  n_samples       split
0           SimCLR         Rotation    0.248448       2533  Validation
1           SimCLR           DINOv2    0.606945       2533  Validation
2         Rotation           DINOv2    0.145899       2533  Validation

In [8]:
# ============================================================
# CELL 7 — CKA PERMUTATION SIGNIFICANCE TEST
# ============================================================

rng = np.random.default_rng(
    RANDOM_STATE
)

subset_n = min(
    CKA_PERMUTATION_SUBSET,
    len(Y_VAL),
)

# Stratified-ish fixed subset: sample approximately in proportion to class size.
selected = []

for c in range(NUM_CLASSES):
    idx = np.flatnonzero(
        Y_VAL == c
    )

    n_c = max(
        1,
        int(
            round(
                subset_n
                * len(idx)
                / len(Y_VAL)
            )
        )
    )

    n_c = min(
        n_c,
        len(idx),
    )

    selected.extend(
        rng.choice(
            idx,
            size=n_c,
            replace=False,
        ).tolist()
    )

selected = np.asarray(
    selected,
    dtype=np.int64,
)

# Enforce exact maximum size if rounding overshoots.
if len(selected) > subset_n:
    selected = rng.choice(
        selected,
        size=subset_n,
        replace=False,
    )

perm_rows = []

for a, b in pair_names:
    xa = REPRESENTATIONS[a][selected]
    xb = REPRESENTATIONS[b][selected]

    ka = center_gram(
        linear_gram(xa)
    )

    kb = center_gram(
        linear_gram(xb)
    )

    observed = cka_from_centered_grams(
        ka,
        kb,
    )

    null_scores = np.empty(
        CKA_PERMUTATIONS,
        dtype=np.float64,
    )

    n = len(selected)

    for r in range(
        CKA_PERMUTATIONS
    ):
        perm = rng.permutation(
            n
        )

        kb_perm = kb[
            np.ix_(
                perm,
                perm,
            )
        ]

        null_scores[r] = (
            cka_from_centered_grams(
                ka,
                kb_perm,
            )
        )

    p_value = (
        1
        + np.sum(
            null_scores
            >= observed
        )
    ) / (
        CKA_PERMUTATIONS
        + 1
    )

    perm_rows.append({
        "representation_a": a,
        "representation_b": b,
        "observed_cka_subset": observed,
        "null_mean": float(
            np.mean(
                null_scores
            )
        ),
        "null_std": float(
            np.std(
                null_scores,
                ddof=1,
            )
        ),
        "permutation_p": float(
            p_value
        ),
        "subset_n": int(
            len(selected)
        ),
        "permutations": CKA_PERMUTATIONS,
        "seed": RANDOM_STATE,
    })

    print(
        f"{a} ↔ {b}: "
        f"observed={observed:.5f}, "
        f"null={np.mean(null_scores):.5f}, "
        f"p={p_value:.5f}"
    )

CKA_PERM_DF = pd.DataFrame(
    perm_rows
)

display(CKA_PERM_DF)

CKA_PERM_DF.to_csv(
    TABLE_DIR / "cka_permutation_significance.csv",
    index=False,
)

SimCLR ↔ Rotation: observed=0.24399, null=0.00883, p=0.00498
SimCLR ↔ DINOv2: observed=0.61526, null=0.03352, p=0.00498
Rotation ↔ DINOv2: observed=0.14332, null=0.00564, p=0.00498


  representation_a representation_b  observed_cka_subset  null_mean  null_std  \
0           SimCLR         Rotation             0.243988   0.008830  0.001757   
1           SimCLR           DINOv2             0.615264   0.033523  0.001612   
2         Rotation           DINOv2             0.143325   0.005643  0.001858   

   permutation_p  subset_n  permutations  seed  
0       0.004975       800           200    42  
1       0.004975       800           200    42  
2       0.004975       800           200    42  

In [9]:
# ============================================================
# CELL 8 — REPRESENTATION QUALITY METRICS
# ============================================================

def reduce_for_quality(
    features,
    max_dim=QUALITY_PCA_DIM,
):
    x = np.asarray(
        features,
        dtype=np.float32,
    )

    x = StandardScaler(
        with_mean=True,
        with_std=True,
    ).fit_transform(x)

    dim = min(
        max_dim,
        x.shape[1],
        x.shape[0] - 1,
    )

    if dim < x.shape[1]:
        x = PCA(
            n_components=dim,
            random_state=RANDOM_STATE,
        ).fit_transform(x)

    return x.astype(
        np.float32
    )

def knn_label_purity(
    x,
    y,
    k=KNN_K,
):
    nn = NearestNeighbors(
        n_neighbors=k + 1,
        metric="euclidean",
    )

    nn.fit(x)

    indices = nn.kneighbors(
        x,
        return_distance=False,
    )[:, 1:]

    neighbor_labels = y[
        indices
    ]

    purity = np.mean(
        neighbor_labels
        == y[:, None]
    )

    return float(purity)

def fisher_ratio(
    x,
    y,
):
    global_mean = x.mean(
        axis=0
    )

    between = 0.0
    within = 0.0

    for c in range(
        NUM_CLASSES
    ):
        xc = x[
            y == c
        ]

        if len(xc) == 0:
            continue

        mean_c = xc.mean(
            axis=0
        )

        between += (
            len(xc)
            * np.sum(
                (
                    mean_c
                    - global_mean
                ) ** 2
            )
        )

        within += np.sum(
            (
                xc
                - mean_c
            ) ** 2
        )

    return float(
        between
        / max(
            within,
            1e-12,
        )
    )

quality_rows = []

for name, features in REPRESENTATIONS.items():
    xq = reduce_for_quality(
        features
    )

    silhouette = float(
        silhouette_score(
            xq,
            Y_VAL,
            metric="euclidean",
        )
    )

    purity = knn_label_purity(
        xq,
        Y_VAL,
        KNN_K,
    )

    fisher = fisher_ratio(
        xq,
        Y_VAL,
    )

    quality_rows.append({
        "representation": name,
        "silhouette": silhouette,
        "knn10_label_purity": purity,
        "fisher_ratio": fisher,
        "quality_dim": xq.shape[1],
        "split": "Validation",
    })

    print(
        name,
        "| silhouette",
        f"{silhouette:.4f}",
        "| kNN purity",
        f"{purity:.4f}",
        "| Fisher",
        f"{fisher:.4f}",
    )

QUALITY_DF = pd.DataFrame(
    quality_rows
)

display(QUALITY_DF)

QUALITY_DF.to_csv(
    TABLE_DIR / "representation_quality_metrics.csv",
    index=False,
)

SimCLR | silhouette 0.0518 | kNN purity 0.6619 | Fisher 0.1491
Rotation | silhouette -0.1306 | kNN purity 0.4277 | Fisher 0.0460
DINOv2 | silhouette 0.0758 | kNN purity 0.6915 | Fisher 0.2882


  representation  silhouette  knn10_label_purity  fisher_ratio  quality_dim  \
0         SimCLR    0.051840            0.661942      0.149054           64   
1       Rotation   -0.130565            0.427714      0.045993           64   
2         DINOv2    0.075838            0.691473      0.288226           64   

        split  
0  Validation  
1  Validation  
2  Validation  

In [10]:
# ============================================================
# CELL 9 — UMAP VISUALIZATIONS
# ============================================================

def make_umap_embedding(
    features,
    model_name,
):
    x = np.asarray(
        features,
        dtype=np.float32,
    )

    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(
        x
    )

    pca_dim = min(
        UMAP_PCA_DIM,
        x_scaled.shape[1],
        x_scaled.shape[0] - 1,
    )

    if pca_dim < x_scaled.shape[1]:
        x_reduced = PCA(
            n_components=pca_dim,
            random_state=RANDOM_STATE,
        ).fit_transform(
            x_scaled
        )
    else:
        x_reduced = x_scaled

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=30,
        min_dist=0.10,
        metric="euclidean",
        random_state=RANDOM_STATE,
    )

    emb = reducer.fit_transform(
        x_reduced
    )

    emb_df = pd.DataFrame({
        "x": emb[:, 0],
        "y": emb[:, 1],
        "label": Y_VAL,
        "class_name": [
            CLASS_NAMES[int(v)]
            for v in Y_VAL
        ],
        "representation": model_name,
    })

    emb_df.to_csv(
        EMBED_DIR
        / f"{model_name.lower()}_validation_umap.csv",
        index=False,
    )

    return emb_df

UMAP_RESULTS = {}

for model_name, features in REPRESENTATIONS.items():
    print("UMAP:", model_name)

    emb_df = make_umap_embedding(
        features,
        model_name,
    )

    UMAP_RESULTS[
        model_name
    ] = emb_df

    plt.figure(
        figsize=(7.4, 6.2)
    )

    for c, class_name in enumerate(
        CLASS_NAMES
    ):
        part = emb_df.loc[
            emb_df["label"] == c
        ]

        plt.scatter(
            part["x"],
            part["y"],
            s=8,
            alpha=0.65,
            label=class_name,
        )

    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")

    plt.title(
        f"{model_name} Representation — Original Validation Split"
    )

    plt.legend(
        markerscale=2,
        fontsize=8,
        ncol=2,
    )

    plt.tight_layout()

    plt.savefig(
        FIG_DIR
        / f"{model_name.lower()}_validation_umap.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

gc.collect()

UMAP: SimCLR


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP: Rotation


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP: DINOv2


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


97450

In [11]:
# ============================================================
# CELL 10 — LOAD SAME-SPLIT PREDICTIONS
# ============================================================

# ---- Final proposed probability
final_hits = find_all(
    "final_internal_probabilities.npz"
)

FINAL_NPZ = None

for p in sorted(
    final_hits,
    key=lambda p: (
        0 if "reuse_only_all_model_ensemble" in str(p) else 9,
        len(p.parts),
        str(p),
    )
):
    try:
        with np.load(p) as z:
            if not {
                "y_test",
                "final_test_prob",
            }.issubset(z.files):
                continue

            yt = np.asarray(
                z["y_test"],
                dtype=np.int32,
            )

            if not np.array_equal(
                yt,
                Y_TEST,
            ):
                continue

        FINAL_NPZ = p
        break

    except Exception:
        continue

if FINAL_NPZ is None:
    raise FileNotFoundError(
        "Frozen final original-split probability file not found."
    )

with np.load(FINAL_NPZ) as z:
    FINAL_TEST_PROB = normalize_probabilities(
        z["final_test_prob"]
    )

# ---- Identify original pipeline root using SimCLR
sim_hits = find_all(
    "multiseed_probabilities_no_tta.npz"
)

SIMCLR_NPZ = None
PIPELINE_ROOT = None

for p in sim_hits:
    try:
        with np.load(p) as z:
            if not {
                "y_test",
                "test_prob",
            }.issubset(z.files):
                continue

            yt = np.asarray(
                z["y_test"],
                dtype=np.int32,
            )

            if not np.array_equal(
                yt,
                Y_TEST,
            ):
                continue

        SIMCLR_NPZ = p
        PIPELINE_ROOT = p.parent.parent
        break

    except Exception:
        continue

if SIMCLR_NPZ is None:
    raise FileNotFoundError(
        "Original SimCLR same-split probability file not found."
    )

with np.load(SIMCLR_NPZ) as z:
    SIMCLR_TEST_PROB = normalize_probabilities(
        z["test_prob"]
    )

# ---- Rotation
ROT_TEST_CSV = (
    PIPELINE_ROOT
    / "extended_imagenet_rotation_finetune"
    / "imagenet_rotation_test_predictions.csv"
)

rotation_y, ROTATION_TEST_PROB = load_prediction_csv(
    ROT_TEST_CSV
)

if not np.array_equal(
    rotation_y,
    Y_TEST,
):
    raise RuntimeError(
        "Rotation test labels are not aligned."
    )

# ---- DINO multiseed prediction, not representative seed
dino_prob_hits = find_all(
    "dinov2_original_split_multiseed_probabilities.npz"
)

DINO_PROB_NPZ = None

for p in dino_prob_hits:
    try:
        with np.load(p) as z:
            if not {
                "y_test",
                "test_prob",
            }.issubset(z.files):
                continue

            yt = np.asarray(
                z["y_test"],
                dtype=np.int32,
            )

            if not np.array_equal(
                yt,
                Y_TEST,
            ):
                continue

        DINO_PROB_NPZ = p
        break

    except Exception:
        continue

if DINO_PROB_NPZ is None:
    raise FileNotFoundError(
        "J2 DINO multiseed probability file not found."
    )

with np.load(DINO_PROB_NPZ) as z:
    DINO_TEST_PROB = normalize_probabilities(
        z["test_prob"]
    )

PREDICTIONS = {
    "SimCLR": np.argmax(
        SIMCLR_TEST_PROB,
        axis=1,
    ),
    "Rotation": np.argmax(
        ROTATION_TEST_PROB,
        axis=1,
    ),
    "DINOv2": np.argmax(
        DINO_TEST_PROB,
        axis=1,
    ),
    "Final Proposed": np.argmax(
        FINAL_TEST_PROB,
        axis=1,
    ),
}

print("Prediction sources loaded and aligned.")

Prediction sources loaded and aligned.


In [12]:
# ============================================================
# CELL 11 — PAIRWISE PREDICTION COMPLEMENTARITY
# ============================================================

def pair_complementarity(
    pred_a,
    pred_b,
    y_true,
):
    correct_a = (
        pred_a == y_true
    )

    correct_b = (
        pred_b == y_true
    )

    n11 = int(
        np.sum(
            correct_a
            & correct_b
        )
    )

    n10 = int(
        np.sum(
            correct_a
            & ~correct_b
        )
    )

    n01 = int(
        np.sum(
            ~correct_a
            & correct_b
        )
    )

    n00 = int(
        np.sum(
            ~correct_a
            & ~correct_b
        )
    )

    denominator = (
        n11 * n00
        + n10 * n01
    )

    q_stat = (
        np.nan
        if denominator == 0
        else (
            n11 * n00
            - n10 * n01
        ) / denominator
    )

    disagreement = float(
        np.mean(
            pred_a
            != pred_b
        )
    )

    double_fault = (
        n00
        / len(y_true)
    )

    return {
        "both_correct": n11,
        "a_only_correct": n10,
        "b_only_correct": n01,
        "both_wrong": n00,
        "disagreement_rate": disagreement,
        "q_statistic": float(q_stat),
        "double_fault_rate": float(double_fault),
    }

pairwise_rows = []

prediction_names = [
    "SimCLR",
    "Rotation",
    "DINOv2",
    "Final Proposed",
]

for i in range(
    len(prediction_names)
):
    for j in range(
        i + 1,
        len(prediction_names)
    ):
        a = prediction_names[i]
        b = prediction_names[j]

        stats = pair_complementarity(
            PREDICTIONS[a],
            PREDICTIONS[b],
            Y_TEST,
        )

        pairwise_rows.append({
            "model_a": a,
            "model_b": b,
            **stats,
        })

PAIRWISE_PRED_DF = pd.DataFrame(
    pairwise_rows
)

display(PAIRWISE_PRED_DF)

PAIRWISE_PRED_DF.to_csv(
    TABLE_DIR / "pairwise_prediction_complementarity.csv",
    index=False,
)

    model_a         model_b  both_correct  a_only_correct  b_only_correct  \
0    SimCLR        Rotation          1878             262             137   
1    SimCLR          DINOv2          1868             272             150   
2    SimCLR  Final Proposed          2072              68              54   
3  Rotation          DINOv2          1799             216             219   
4  Rotation  Final Proposed          1898             117             228   
5    DINOv2  Final Proposed          1866             152             260   

   both_wrong  disagreement_rate  q_statistic  double_fault_rate  
0         257           0.177190     0.861557           0.101421  
1         244           0.187056     0.835680           0.096290  
2         340           0.062747     0.989629           0.134175  
3         300           0.200079     0.838829           0.118390  
4         291           0.155880     0.907854           0.114838  
5         256           0.186267     0.847182           0.

In [13]:
# ============================================================
# CELL 12 — ERROR RESCUE BY THE FINAL MODEL
# ============================================================

final_pred = PREDICTIONS[
    "Final Proposed"
]

final_correct = (
    final_pred == Y_TEST
)

rescue_rows = []

for source_name in [
    "SimCLR",
    "Rotation",
    "DINOv2",
]:
    source_pred = PREDICTIONS[
        source_name
    ]

    source_correct = (
        source_pred == Y_TEST
    )

    source_errors = ~source_correct

    rescued = (
        source_errors
        & final_correct
    )

    lost = (
        source_correct
        & ~final_correct
    )

    rescue_rows.append({
        "source_model": source_name,
        "source_total_errors": int(
            np.sum(
                source_errors
            )
        ),
        "errors_rescued_by_final": int(
            np.sum(
                rescued
            )
        ),
        "rescue_rate_among_source_errors": float(
            np.sum(
                rescued
            )
            / max(
                np.sum(
                    source_errors
                ),
                1,
            )
        ),
        "source_correct_but_final_wrong": int(
            np.sum(
                lost
            )
        ),
        "net_correctness_gain": int(
            np.sum(
                rescued
            )
            - np.sum(
                lost
            )
        ),
    })

RESCUE_DF = pd.DataFrame(
    rescue_rows
)

display(RESCUE_DF)

RESCUE_DF.to_csv(
    TABLE_DIR / "final_model_error_rescue_summary.csv",
    index=False,
)

  source_model  source_total_errors  errors_rescued_by_final  \
0       SimCLR                  394                       54   
1     Rotation                  519                      228   
2       DINOv2                  516                      260   

   rescue_rate_among_source_errors  source_correct_but_final_wrong  \
0                         0.137056                              68   
1                         0.439306                             117   
2                         0.503876                             152   

   net_correctness_gain  
0                   -14  
1                   111  
2                   108  

In [14]:
# ============================================================
# CELL 13 — CLASS-WISE UNIQUE CORRECT / RESCUE ANALYSIS
# ============================================================

class_rows = []

for source_name in [
    "SimCLR",
    "Rotation",
    "DINOv2",
]:
    source_pred = PREDICTIONS[
        source_name
    ]

    source_correct = (
        source_pred == Y_TEST
    )

    for c, class_name in enumerate(
        CLASS_NAMES
    ):
        mask = (
            Y_TEST == c
        )

        source_unique_vs_final = int(
            np.sum(
                mask
                & source_correct
                & ~final_correct
            )
        )

        final_rescue = int(
            np.sum(
                mask
                & ~source_correct
                & final_correct
            )
        )

        class_rows.append({
            "source_model": source_name,
            "class_id": c,
            "class_name": class_name,
            "class_n": int(
                np.sum(mask)
            ),
            "source_correct_final_wrong": source_unique_vs_final,
            "source_wrong_final_correct": final_rescue,
            "net_final_rescue": (
                final_rescue
                - source_unique_vs_final
            ),
        })

CLASS_RESCUE_DF = pd.DataFrame(
    class_rows
)

display(CLASS_RESCUE_DF)

CLASS_RESCUE_DF.to_csv(
    TABLE_DIR / "classwise_final_error_rescue.csv",
    index=False,
)

# Rotation unique-correct against SimCLR specifically
sim_correct = (
    PREDICTIONS["SimCLR"]
    == Y_TEST
)

rot_correct = (
    PREDICTIONS["Rotation"]
    == Y_TEST
)

rotation_unique_rows = []

for c, class_name in enumerate(
    CLASS_NAMES
):
    mask = (
        Y_TEST == c
    )

    rotation_unique_rows.append({
        "class_id": c,
        "class_name": class_name,
        "class_n": int(
            np.sum(mask)
        ),
        "rotation_correct_simclr_wrong": int(
            np.sum(
                mask
                & rot_correct
                & ~sim_correct
            )
        ),
        "simclr_correct_rotation_wrong": int(
            np.sum(
                mask
                & sim_correct
                & ~rot_correct
            )
        ),
    })

ROTATION_UNIQUE_DF = pd.DataFrame(
    rotation_unique_rows
)

display(ROTATION_UNIQUE_DF)

ROTATION_UNIQUE_DF.to_csv(
    TABLE_DIR / "rotation_simclr_classwise_unique_correct.csv",
    index=False,
)

   source_model  class_id class_name  class_n  source_correct_final_wrong  \
0        SimCLR         0        MEL      452                          15   
1        SimCLR         1         NV     1288                          30   
2        SimCLR         2        BCC      333                           9   
3        SimCLR         3         AK       86                           4   
4        SimCLR         4        BKL      263                          10   
5        SimCLR         5         DF       24                           0   
6        SimCLR         6       VASC       25                           0   
7        SimCLR         7        SCC       63                           0   
8      Rotation         0        MEL      452                          21   
9      Rotation         1         NV     1288                          57   
10     Rotation         2        BCC      333                          20   
11     Rotation         3         AK       86                           2   

   class_id class_name  class_n  rotation_correct_simclr_wrong  \
0         0        MEL      452                             25   
1         1         NV     1288                             59   
2         2        BCC      333                             22   
3         3         AK       86                              5   
4         4        BKL      263                             17   
5         5         DF       24                              2   
6         6       VASC       25                              0   
7         7        SCC       63                              7   

   simclr_correct_rotation_wrong  
0                             68  
1                             77  
2                             40  
3                             18  
4                             42  
5                              3  
6                              1  
7                             13  

In [15]:
# ============================================================
# CELL 14 — COMPLEMENTARITY MATRIX FIGURES
# ============================================================

# Disagreement matrix
names = [
    "SimCLR",
    "Rotation",
    "DINOv2",
    "Final Proposed",
]

disagreement_matrix = np.zeros(
    (
        len(names),
        len(names),
    ),
    dtype=np.float64,
)

for i, a in enumerate(
    names
):
    for j, b in enumerate(
        names
    ):
        disagreement_matrix[
            i,
            j
        ] = np.mean(
            PREDICTIONS[a]
            != PREDICTIONS[b]
        )

plt.figure(
    figsize=(6.8, 5.8)
)

plt.imshow(
    100 * disagreement_matrix,
    interpolation="nearest",
)

plt.colorbar(
    label="Disagreement (%)"
)

ticks = np.arange(
    len(names)
)

plt.xticks(
    ticks,
    names,
    rotation=35,
    ha="right",
)

plt.yticks(
    ticks,
    names,
)

for i in range(
    len(names)
):
    for j in range(
        len(names)
    ):
        plt.text(
            j,
            i,
            f"{100 * disagreement_matrix[i,j]:.1f}",
            ha="center",
            va="center",
        )

plt.title(
    "Prediction Disagreement — Original Internal Test"
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "prediction_disagreement_matrix.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

pd.DataFrame(
    disagreement_matrix,
    index=names,
    columns=names,
).to_csv(
    TABLE_DIR / "prediction_disagreement_matrix.csv"
)

In [16]:
# ============================================================
# CELL 15 — CKA MATRIX FIGURE
# ============================================================

rep_names = [
    "SimCLR",
    "Rotation",
    "DINOv2",
]

cka_matrix = np.eye(
    len(rep_names),
    dtype=np.float64,
)

for _, row in CKA_DF.iterrows():
    i = rep_names.index(
        row["representation_a"]
    )

    j = rep_names.index(
        row["representation_b"]
    )

    cka_matrix[
        i,
        j
    ] = row["linear_cka"]

    cka_matrix[
        j,
        i
    ] = row["linear_cka"]

plt.figure(
    figsize=(6.4, 5.4)
)

plt.imshow(
    cka_matrix,
    vmin=0,
    vmax=1,
    interpolation="nearest",
)

plt.colorbar(
    label="Linear CKA"
)

ticks = np.arange(
    len(rep_names)
)

plt.xticks(
    ticks,
    rep_names,
)

plt.yticks(
    ticks,
    rep_names,
)

for i in range(
    len(rep_names)
):
    for j in range(
        len(rep_names)
    ):
        plt.text(
            j,
            i,
            f"{cka_matrix[i,j]:.3f}",
            ha="center",
            va="center",
        )

plt.title(
    "Representation Similarity — Validation Split"
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "pairwise_cka_matrix.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

pd.DataFrame(
    cka_matrix,
    index=rep_names,
    columns=rep_names,
).to_csv(
    TABLE_DIR / "pairwise_cka_matrix.csv"
)

In [17]:
# ============================================================
# CELL 16 — JOURNAL INTERPRETATION AUDIT
# ============================================================

sim_rot_row = PAIRWISE_PRED_DF.loc[
    (
        PAIRWISE_PRED_DF["model_a"]
        == "SimCLR"
    )
    &
    (
        PAIRWISE_PRED_DF["model_b"]
        == "Rotation"
    )
].iloc[0]

interpretation = {
    "protocol":
        "original stratified 80/10/10",

    "validation_n":
        len(Y_VAL),

    "test_n":
        len(Y_TEST),

    "representations": {
        "SimCLR_dim":
            int(
                SIM_VAL_FEATURES.shape[1]
            ),

        "Rotation_dim":
            int(
                ROT_VAL_FEATURES.shape[1]
            ),

        "DINOv2_dim":
            int(
                DINO_VAL_FEATURES.shape[1]
            ),

        "DINO_representative_seed":
            DINO_REP_SEED,
    },

    "pairwise_cka":
        CKA_DF.to_dict(
            orient="records"
        ),

    "cka_permutation":
        CKA_PERM_DF.to_dict(
            orient="records"
        ),

    "representation_quality":
        QUALITY_DF.to_dict(
            orient="records"
        ),

    "simclr_rotation_prediction_complementarity":
        sim_rot_row.to_dict(),

    "final_error_rescue":
        RESCUE_DF.to_dict(
            orient="records"
        ),

    "interpretation_rule": (
        "A lower CKA between two competent representations supports "
        "representational diversity, but lower CKA alone does not imply "
        "better classification. Prediction disagreement, unique-correct "
        "cases, and final-model error rescue are reported together to "
        "support the heterogeneous-SSL complementarity claim."
    ),
}

with open(
    AUDIT_DIR / "J5_complementarity_summary.json",
    "w",
) as f:
    json.dump(
        interpretation,
        f,
        indent=2,
        default=float,
    )

print(
    json.dumps(
        interpretation,
        indent=2,
        default=float,
    )
)

{
  "protocol": "original stratified 80/10/10",
  "validation_n": 2533,
  "test_n": 2534,
  "representations": {
    "SimCLR_dim": 2048,
    "Rotation_dim": 2048,
    "DINOv2_dim": 384,
    "DINO_representative_seed": 42
  },
  "pairwise_cka": [
    {
      "representation_a": "SimCLR",
      "representation_b": "Rotation",
      "linear_cka": 0.24844778643860746,
      "n_samples": 2533,
      "split": "Validation"
    },
    {
      "representation_a": "SimCLR",
      "representation_b": "DINOv2",
      "linear_cka": 0.606944618629552,
      "n_samples": 2533,
      "split": "Validation"
    },
    {
      "representation_a": "Rotation",
      "representation_b": "DINOv2",
      "linear_cka": 0.14589861678816984,
      "n_samples": 2533,
      "split": "Validation"
    }
  ],
  "cka_permutation": [
    {
      "representation_a": "SimCLR",
      "representation_b": "Rotation",
      "observed_cka_subset": 0.2439882291029237,
      "null_mean": 0.008830239272766693,
      "null_std": 

In [18]:
# ============================================================
# CELL 17 — COMPLETION MARKER + ZIP
# ============================================================

marker = {
    "stage":
        "J5_original_split_representation_complementarity",

    "status":
        "complete",

    "protocol":
        "original_stratified_80_10_10",

    "validation_n":
        len(Y_VAL),

    "test_n":
        len(Y_TEST),

    "dino_representative_seed":
        DINO_REP_SEED,

    "cka_permutations":
        CKA_PERMUTATIONS,

    "test_used_for_selection":
        False,

    "output_rel":
        "j5_original_split_complementarity",
}

MARKER_PATH = (
    WORKING
    / "J5_COMPLEMENTARITY_COMPLETE.json"
)

with open(
    MARKER_PATH,
    "w",
) as f:
    json.dump(
        marker,
        f,
        indent=2,
    )

shutil.copy2(
    MARKER_PATH,
    OUTPUT_DIR
    / "J5_COMPLEMENTARITY_COMPLETE.json",
)

ZIP_PATH = (
    WORKING
    / "J5_ORIGINAL_SPLIT_COMPLEMENTARITY_RESULTS.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for p in sorted(
        OUTPUT_DIR.rglob("*")
    ):
        if p.is_file():
            zf.write(
                p,
                arcname=str(
                    p.relative_to(
                        OUTPUT_DIR.parent
                    )
                ),
            )

required = [
    TABLE_DIR
    / "pairwise_linear_cka.csv",

    TABLE_DIR
    / "cka_permutation_significance.csv",

    TABLE_DIR
    / "representation_quality_metrics.csv",

    TABLE_DIR
    / "pairwise_prediction_complementarity.csv",

    TABLE_DIR
    / "final_model_error_rescue_summary.csv",

    TABLE_DIR
    / "classwise_final_error_rescue.csv",

    FIG_DIR
    / "pairwise_cka_matrix.png",

    FIG_DIR
    / "prediction_disagreement_matrix.png",

    FIG_DIR
    / "simclr_validation_umap.png",

    FIG_DIR
    / "rotation_validation_umap.png",

    FIG_DIR
    / "dinov2_validation_umap.png",

    AUDIT_DIR
    / "J5_complementarity_summary.json",

    MARKER_PATH,

    ZIP_PATH,
]

missing = [
    str(p)
    for p in required
    if not p.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing J5 outputs:\n"
        + "\n".join(
            missing
        )
    )

print("=" * 96)
print("J5 REPRESENTATION / COMPLEMENTARITY COMPLETE")
print("=" * 96)

print("\nCKA:")
display(CKA_DF)

print("\nRepresentation quality:")
display(QUALITY_DF)

print("\nFinal-model error rescue:")
display(RESCUE_DF)

print("\nZIP:")
print(ZIP_PATH)

print()
print(
    "Save this Kaggle notebook Version WITH OUTPUTS "
    "and send the executed notebook back."
)

J5 REPRESENTATION / COMPLEMENTARITY COMPLETE

CKA:


  representation_a representation_b  linear_cka  n_samples       split
0           SimCLR         Rotation    0.248448       2533  Validation
1           SimCLR           DINOv2    0.606945       2533  Validation
2         Rotation           DINOv2    0.145899       2533  Validation


Representation quality:


  representation  silhouette  knn10_label_purity  fisher_ratio  quality_dim  \
0         SimCLR    0.051840            0.661942      0.149054           64   
1       Rotation   -0.130565            0.427714      0.045993           64   
2         DINOv2    0.075838            0.691473      0.288226           64   

        split  
0  Validation  
1  Validation  
2  Validation  


Final-model error rescue:


  source_model  source_total_errors  errors_rescued_by_final  \
0       SimCLR                  394                       54   
1     Rotation                  519                      228   
2       DINOv2                  516                      260   

   rescue_rate_among_source_errors  source_correct_but_final_wrong  \
0                         0.137056                              68   
1                         0.439306                             117   
2                         0.503876                             152   

   net_correctness_gain  
0                   -14  
1                   111  
2                   108  


ZIP:
/kaggle/working/J5_ORIGINAL_SPLIT_COMPLEMENTARITY_RESULTS.zip

Save this Kaggle notebook Version WITH OUTPUTS and send the executed notebook back.
